# Prompt Testing Framework: Systematic Testing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/12-meta-prompting/101_prompt_testing_framework.ipynb)

**Category**: 12 - Meta-Prompting | **Technique #101**

---

A systematic framework for testing prompts against defined criteria, edge cases, and expected outputs to ensure reliability and quality before deployment.

## Description

Prompt Testing Frameworks provide:
- Automated test execution
- Regression detection
- Edge case coverage
- Performance benchmarking
- Quality gates for deployment

**When to use:**
- Before deploying prompts to production
- When modifying existing prompts
- For continuous integration pipelines
- When comparing prompt versions
- For compliance verification

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                 PROMPT TESTING FRAMEWORK                    │
└─────────────────────────────────────────────────────────────┘

  Test Suite Structure
  ┌─────────────────────────────────────────────────────┐
  │  Prompt Under Test                                  │
  │         │                                           │
  │    ┌────┴────┬────────┬────────┬────────┐          │
  │    ▼         ▼        ▼        ▼        ▼          │
  │  Test 1   Test 2   Test 3   Test 4   Test N        │
  │    │         │        │        │        │          │
  │    └────────┴────────┴────────┴────────┘          │
  │                      │                             │
  │                      ▼                             │
  │              ┌──────────────┐                      │
  │              │   Results    │                      │
  │              │  ✓ ✓ ✗ ✓ ✓   │                      │
  │              └──────────────┘                      │
  └─────────────────────────────────────────────────────┘

  Test Types:
  ├── Unit Tests: Individual prompt behavior
  ├── Integration Tests: End-to-end workflows
  ├── Edge Case Tests: Boundary conditions
  ├── Regression Tests: Prevent degradation
  └── Performance Tests: Latency, token usage
```

## Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
import json
import time
import re
from typing import Dict, List, Any, Callable, Optional
from dataclasses import dataclass, field
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## Basic Example: Simple Test Framework

In [ ]:
@dataclass
class TestResult:
    """Result of a single test."""
    name: str
    passed: bool
    input_data: str
    expected: str
    actual: str
    message: str
    duration_ms: float
    tokens_used: int = 0

class PromptTest:
    """Individual prompt test."""
    
    def __init__(
        self, 
        name: str, 
        input_data: str, 
        assertion: Callable[[str], tuple[bool, str]],
        expected: str = ""
    ):
        self.name = name
        self.input_data = input_data
        self.assertion = assertion
        self.expected = expected

class PromptTestSuite:
    """Suite of tests for a prompt."""
    
    def __init__(self, prompt_template: str, model: str = "gpt-4o"):
        self.prompt_template = prompt_template
        self.model = model
        self.tests: List[PromptTest] = []
        self.results: List[TestResult] = []
    
    def add_test(self, test: PromptTest):
        """Add a test to the suite."""
        self.tests.append(test)
    
    def run(self, client: OpenAI) -> Dict[str, Any]:
        """Run all tests."""
        self.results = []
        
        for test in self.tests:
            start_time = time.time()
            
            # Format prompt with test input
            prompt = self.prompt_template.format(input=test.input_data)
            
            # Call LLM
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            
            actual = response.choices[0].message.content
            tokens = response.usage.total_tokens if response.usage else 0
            duration = (time.time() - start_time) * 1000
            
            # Run assertion
            passed, message = test.assertion(actual)
            
            result = TestResult(
                name=test.name,
                passed=passed,
                input_data=test.input_data,
                expected=test.expected,
                actual=actual,
                message=message,
                duration_ms=duration,
                tokens_used=tokens
            )
            self.results.append(result)
        
        return self._generate_report()
    
    def _generate_report(self) -> Dict[str, Any]:
        """Generate test report."""
        passed = sum(1 for r in self.results if r.passed)
        total = len(self.results)
        
        return {
            "total_tests": total,
            "passed": passed,
            "failed": total - passed,
            "pass_rate": passed / total if total > 0 else 0,
            "avg_duration_ms": sum(r.duration_ms for r in self.results) / total if total > 0 else 0,
            "total_tokens": sum(r.tokens_used for r in self.results),
            "results": [
                {
                    "name": r.name,
                    "passed": r.passed,
                    "message": r.message,
                    "duration_ms": round(r.duration_ms, 2)
                }
                for r in self.results
            ]
        }

# Create a prompt and test suite
sentiment_prompt = """
Analyze the sentiment of the following text.
Respond with ONLY ONE of: POSITIVE, NEGATIVE, or NEUTRAL.

Text: {input}
Sentiment:"""

suite = PromptTestSuite(sentiment_prompt)

# Add tests
suite.add_test(PromptTest(
    name="positive_sentiment",
    input_data="I love this product! It's amazing!",
    assertion=lambda x: ("POSITIVE" in x.upper(), "Expected POSITIVE sentiment"),
    expected="POSITIVE"
))

suite.add_test(PromptTest(
    name="negative_sentiment",
    input_data="This is terrible. I hate it.",
    assertion=lambda x: ("NEGATIVE" in x.upper(), "Expected NEGATIVE sentiment"),
    expected="NEGATIVE"
))

suite.add_test(PromptTest(
    name="neutral_sentiment",
    input_data="The product arrived yesterday.",
    assertion=lambda x: ("NEUTRAL" in x.upper(), "Expected NEUTRAL sentiment"),
    expected="NEUTRAL"
))

suite.add_test(PromptTest(
    name="empty_input",
    input_data="",
    assertion=lambda x: (len(x) > 0, "Should handle empty input gracefully"),
    expected="NEUTRAL or error message"
))

# Run tests
report = suite.run(client)

print("=== TEST REPORT ===\n")
print(f"Total Tests: {report['total_tests']}")
print(f"Passed: {report['passed']} ✓")
print(f"Failed: {report['failed']} ✗")
print(f"Pass Rate: {report['pass_rate']*100:.1f}%")
print(f"Avg Duration: {report['avg_duration_ms']:.0f}ms")
print(f"Total Tokens: {report['total_tokens']}")
print("\n=== DETAILED RESULTS ===")
for result in report['results']:
    status = "✓" if result['passed'] else "✗"
    print(f"{status} {result['name']}: {result['message']} ({result['duration_ms']}ms)")

## Real-World Example: Comprehensive Testing Suite

In [ ]:
class ComprehensiveTestFramework:
    """Production-grade testing framework."""
    
    def __init__(self, client: OpenAI):
        self.client = client
        self.suites: Dict[str, PromptTestSuite] = {}
    
    def create_suite(self, name: str, prompt: str, model: str = "gpt-4o") -> PromptTestSuite:
        """Create a new test suite."""
        suite = PromptTestSuite(prompt, model)
        self.suites[name] = suite
        return suite
    
    def run_all(self) -> Dict[str, Any]:
        """Run all test suites."""
        results = {}
        for name, suite in self.suites.items():
            print(f"\nRunning suite: {name}...")
            results[name] = suite.run(self.client)
        return results
    
    def generate_summary(self, results: Dict) -> str:
        """Generate summary report."""
        total_tests = sum(r['total_tests'] for r in results.values())
        total_passed = sum(r['passed'] for r in results.values())
        
        summary = f"""
╔═══════════════════════════════════════════════════════════════╗
║                    TEST SUMMARY REPORT                        ║
╠═══════════════════════════════════════════════════════════════╣
║  Total Suites: {len(results)}                                          ║
║  Total Tests:  {total_tests}                                         ║
║  Passed:       {total_passed} ✓                                      ║
║  Failed:       {total_tests - total_passed} ✗                                        ║
║  Pass Rate:    {total_passed/total_tests*100:.1f}%                                    ║
╚═══════════════════════════════════════════════════════════════╝
"""
        return summary

# Create comprehensive framework
framework = ComprehensiveTestFramework(client)

# Suite 1: Email classifier
email_prompt = """
Classify this email into one category: URGENT, IMPORTANT, or LOW_PRIORITY.

Email: {input}
Category:"""

email_suite = framework.create_suite("email_classifier", email_prompt)

email_suite.add_test(PromptTest(
    "urgent_security",
    "ALERT: Your account has been compromised. Immediate action required.",
    lambda x: ("URGENT" in x.upper(), "Security alerts should be URGENT")
))

email_suite.add_test(PromptTest(
    "important_meeting",
    "Meeting rescheduled to tomorrow at 2pm",
    lambda x: ("IMPORTANT" in x.upper(), "Meeting changes are IMPORTANT")
))

email_suite.add_test(PromptTest(
    "low_newsletter",
    "Weekly newsletter: 10 tips for productivity",
    lambda x: ("LOW" in x.upper() or "PRIORITY" in x.upper(), "Newsletters are LOW_PRIORITY")
))

# Suite 2: JSON extractor
json_prompt = """
Extract information from the text and return as JSON with keys: name, date, amount.

Text: {input}
JSON:"""

json_suite = framework.create_suite("json_extractor", json_prompt)

def is_valid_json(text: str) -> tuple[bool, str]:
    """Check if response is valid JSON."""
    try:
        # Extract JSON if wrapped in code blocks
        if '```json' in text:
            text = text.split('```json')[1].split('```')[0]
        elif '```' in text:
            text = text.split('```')[1].split('```')[0]
        json.loads(text.strip())
        return True, "Valid JSON format"
    except:
        return False, "Invalid JSON format"

json_suite.add_test(PromptTest(
    "simple_extraction",
    "John paid $50 on January 15th",
    is_valid_json
))

json_suite.add_test(PromptTest(
    "complex_extraction",
    "Meeting with Sarah Johnson scheduled for March 3rd, budget approved for $1250.50",
    is_valid_json
))

# Run all suites
all_results = framework.run_all()
print(framework.generate_summary(all_results))

# Detailed results
for suite_name, result in all_results.items():
    print(f"\n{suite_name}:")
    for test in result['results']:
        status = "✓" if test['passed'] else "✗"
        print(f"  {status} {test['name']}")

## Failure Case: Testing Anti-Patterns

In [ ]:
print("=== TESTING ANTI-PATTERNS ===\n")

print("1. BRITTLE ASSERTIONS")
print("   Bad:  assert response == 'The answer is 42.'")
print("   Good: assert '42' in response")
print("   Why:  LLM outputs vary; test for content, not exact match\n")

print("2. INSUFFICIENT COVERAGE")
print("   Bad:  Only testing happy path")
print("   Good: Test edge cases, errors, boundaries")
print("   Why:  Production sees diverse inputs\n")

print("3. NO PERFORMANCE TESTING")
print("   Bad:  Only checking correctness")
print("   Good: Monitor latency, token usage, cost")
print("   Why:  Performance affects user experience\n")

print("4. STATIC TEST DATA")
print("   Bad:  Same inputs every run")
print("   Good: Vary test data, use property-based testing")
print("   Why:  Detects overfitting to specific examples\n")

print("5. IGNORING FLAKINESS")
print("   Bad:  Accepting inconsistent results")
print("   Good: Track flakiness, investigate causes")
print("   Why:  Unreliable tests erode confidence\n")

print("="*60)
print("BEST PRACTICES:")
print("• Test for semantic correctness, not exact matches")
print("• Include negative and edge case tests")
print("• Monitor performance metrics")
print("• Run tests on multiple model versions")
print("• Track test flakiness over time")

## Benchmark: Testing Impact on Quality

| Testing Level | Bug Detection | Production Issues | Confidence |
|---------------|---------------|-------------------|------------|
| None | 20% | High | Low |
| Basic (happy path) | 50% | Medium | Medium |
| Comprehensive | 85% | Low | High |
| Comprehensive + CI | 95% | Very Low | Very High |

**Recommendation**: Comprehensive testing with CI/CD integration.

## Interactive Playground

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE PLAYGROUND                     ║
# ╚═══════════════════════════════════════════════════════════════╝

# Create your own test suite
my_framework = ComprehensiveTestFramework(client)

# Define your prompt
# YOUR_PROMPT = "Your prompt with {input} placeholder"

# Create suite
# my_suite = my_framework.create_suite("my_tests", YOUR_PROMPT)

# Add tests
# my_suite.add_test(PromptTest(
#     name="test_name",
#     input_data="test input",
#     assertion=lambda x: (True, "pass message")
# ))

# Run
# results = my_framework.run_all()
# print(my_framework.generate_summary(results))

## Tips & Tricks

### Assertion Patterns

```python
# Content validation
lambda x: ('keyword' in x.lower(), "Should contain keyword")

# Format validation
lambda x: (x.startswith('{') and x.endswith('}'), "Should be JSON")

# Length validation
lambda x: (50 <= len(x) <= 200, "Should be 50-200 chars")

# Regex validation
lambda x: (re.search(r'\d{3}-\d{4}', x), "Should contain phone number")
```

### CI/CD Integration

```yaml
# .github/workflows/prompt-tests.yml
name: Prompt Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v2
      - name: Run Prompt Tests
        run: python -m pytest tests/prompts/
```

## References

1. [Prompt Engineering Testing Guide](https://platform.openai.com/docs/guides/prompt-engineering)
2. [LangSmith Evaluation](https://docs.smith.langchain.com/evaluation)
3. [Promptfoo Testing Framework](https://promptfoo.dev/)
4. [DeepEval LLM Evaluation](https://docs.confident-ai.com/)

---

**Previous**: [100_prompt_versioning.ipynb](100_prompt_versioning.ipynb) | **Next**: [102_prompt_analytics.ipynb](102_prompt_analytics.ipynb)